# LAB | Connecting Python to SQL

This notebook solves the Sakila rental lab by connecting Python to SQL, retrieving rental records for selected months, counting rentals by customer, and comparing customer activity between May and June 2005.

## 1. Connect Python to the Sakila database

The connection uses SQLAlchemy. Update the username and password if your local MySQL setup uses different credentials.

In [ ]:
from datetime import date

import pandas as pd
from sqlalchemy import create_engine, text

# Change this connection string if your MySQL username or password is different.
connection_string = "mysql+pymysql://root:@localhost:3306/sakila"
engine = create_engine(connection_string)


def month_window(month, year):
    if not 1 <= month <= 12:
        raise ValueError("month must be between 1 and 12")

    start_date = date(year, month, 1)
    if month == 12:
        end_date = date(year + 1, 1, 1)
    else:
        end_date = date(year, month + 1, 1)

    return start_date, end_date


def rentals_month(engine, month, year):
    start_date, end_date = month_window(month, year)

    query = text(
        """
        SELECT *
        FROM rental
        WHERE rental_date >= :start_date
          AND rental_date < :end_date
        ORDER BY rental_date
        """
    )

    return pd.read_sql_query(
        query,
        con=engine,
        params={"start_date": start_date, "end_date": end_date},
    )


def rental_count_month(rentals_df, month, year):
    column_name = f"rentals_{month:02d}_{year}"

    rental_counts = (
        rentals_df.groupby("customer_id")
        .size()
        .reset_index(name=column_name)
        .sort_values("customer_id")
        .reset_index(drop=True)
    )

    return rental_counts


def compare_rentals(first_month_df, second_month_df):
    first_count_column = first_month_df.columns.drop("customer_id")[0]
    second_count_column = second_month_df.columns.drop("customer_id")[0]

    comparison = pd.merge(
        first_month_df,
        second_month_df,
        on="customer_id",
        how="inner",
    )
    comparison["difference"] = (
        comparison[second_count_column] - comparison[first_count_column]
    )

    return comparison.sort_values("customer_id").reset_index(drop=True)

## 2. Retrieve rentals for a selected month

`rentals_month(engine, month, year)` returns all rows from the `rental` table where `rental_date` falls inside the requested month.

In [ ]:
may_rentals = rentals_month(engine, month=5, year=2005)
june_rentals = rentals_month(engine, month=6, year=2005)

display(may_rentals.head())
display(june_rentals.head())

## 3. Count rentals by customer

`rental_count_month(rentals_df, month, year)` groups the monthly rental data by `customer_id` and creates a month-specific count column.

In [ ]:
may_counts = rental_count_month(may_rentals, month=5, year=2005)
june_counts = rental_count_month(june_rentals, month=6, year=2005)

display(may_counts.head())
display(june_counts.head())

## 4. Compare May and June rentals

`compare_rentals(first_month_df, second_month_df)` keeps customers who were active in both months and adds a `difference` column. The difference is calculated as second month rentals minus first month rentals.

In [ ]:
may_june_comparison = compare_rentals(may_counts, june_counts)
may_june_comparison.head()